## Biogas plant

You want to plan the two-year supply of raw materials for a biogas power plant. Such a plant produces energy by burning biogas, which is obtained from the bacterial fermentation of organic wastes. 
Specifically, your plant is powered by corn chopping, a residual of agro-industrial operations that you can purchase from 5 local farms. 
The table below shows the quarterly capacity of each farm for the next two years. Quantities are measured in tons.

Farm|T1|T2|T3|T4|T5|T6|T7|T8
:-|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:
1|700|1500|700|0|0|700|1500|0
2|1350|0|450|0|1350|0|450|0
3|0|1500|1500|0|0|1500|1500|0
4|820|1560|820|0|820|1560|820|0
5|0|680|1080|0|0|680|1080|0

Due to crop rotations and corn harvesting periods, farms are unable to supply material in some quarters. Moreover the types of corn chopping provided are different, each coming with its own unitary purchase price, unitary storage cost and percentage of dry matter. The table below shows a summary of these information.

Farm|Purchase price|Storage cost|Dry matter
:-|:-:|:-:|:-:
1|0.20|0.002|15
2|0.18|0.012|28
3|0.19|0.007|35
4|0.21|0.011|37
5|0.23|0.015|42

Your biogas plant must operate by burning a mixture of corn choppings with a dry matter percentage between 20% and 40%. Under these conditions, the yield is 421.6 kWh of energy per ton of burned material. The energy produced by the plant is sold on the market at a price of 0.28 $/kWh. 

Due to state regulations, all biogas plants can produce a maximum of 1950 MWh of energy per quarter. You are allowed to store corn chopping in a silo, whose total capacity is of 500 tons. 

Plan the supply and inventory of your biogas plant with the goal of maximizing your profits (i.e., revenues minus costs).

## Proposed solution
### Parameters

In [1]:
# Project was done in local, if using collab adapt it
import mip

In [2]:
n = 8 # quarters are indexed from 0 to 7
k = 5 # farms are indexed from 0 to 4

# tons
Q = [
    [700,  1500, 700,  0, 0,    700,  1500, 0],
    [1350, 0,    450,  0, 1350, 0,    450,  0],
    [0,    1500, 1500, 0, 0,    1500, 1500, 0],
    [820,  1560, 820,  0, 820,  1560, 820,  0],
    [0,    680,  1080, 0, 0,    680,  1080, 0]
]

purchase_p = [0.2,   0.18,  0.19,  0.21,  0.23] # per ton
storage_c = [0.002, 0.012, 0.007, 0.011, 0.015] # per ton
dry   = [0.15,  0.28,  0.35,  0.37,  0.42]      # % per ton

min_dry = 0.2 # corn choppings / dry material
max_dry = 0.4 # corn choppings / dry material

_yield = 421.6 # per ton of burned material
price = 0.28   # per kWh

max_Y = 1950000 # kWh
q_silo = 500    # tons

m = mip.Model()

### Variables

In [3]:
# Tons purchased from farms for each q
# Rows are farms while columns are quarters
X = [[m.add_var(var_type=mip.CONTINUOUS) for t in range(n)] for j in range(k)]

# Corn taken from storage for each q
# Rows are farms while columns are quarters
Y = [[m.add_var(var_type=mip.CONTINUOUS) for t in range(n)] for j in range(k)]

# Corn put into storage for each q
# Rows are farms while columns are quarters
Z = [[m.add_var(var_type=mip.CONTINUOUS) for t in range(n)] for j in range(k)]

# Corn effectively burned per q
# Rows are farms while columns are quarters
U = [[m.add_var(var_type=mip.CONTINUOUS) for t in range(n)] for j in range(k)]

# Stored quantity for each q, index is 0..8 (offset of +1)
# Rows are farms while columns are quarters
S = [[m.add_var(var_type=mip.CONTINUOUS) for i in range(n + 1)] for j in range(k)]

# Total energy produced (kWh) per q
W = [m.add_var(var_type=mip.CONTINUOUS) for i in range(n)]

### Constraints

In [4]:
# Initial storage capacity is 0
for i in range(k):
    m.add_constr(S[i][0] == 0)

# Maximum storage capacity is not exceeded for any period
for t in range(n + 1):
    m.add_constr(mip.xsum(S[i][t] for i in range(k)) <= q_silo)

# Multiperiod constraint for storage silo
for t in range(n):
    for i in range(k):
        m.add_constr(S[i][t+1] == S[i][t] + Z[i][t] - Y[i][t])
    
# Purchased quantity is less than what the farm produced
for i in range(k):
    for t in range(n):
        m.add_constr(X[i][t] <= Q[i][t])

# Binding constraint for U
for i in range(k):
    for t in range(n):
        m.add_constr(U[i][t] == X[i][t] + Y[i][t] - Z[i][t])

# Mixture is more than 20%
for t in range(n):
    m.add_constr(mip.xsum(dry[i]*U[i][t] for i in range(k)) - 
                 mip.xsum(min_dry*U[i][t] for i in range(k)) >= 0)

# Mixture is less than 40%
for t in range(n):
    m.add_constr(mip.xsum(dry[i]*U[i][t] for i in range(k)) - 
                 mip.xsum(max_dry*U[i][t] for i in range(k)) <= 0)
    
# Binding constraint for W
for t in range(n):
    m.add_constr(W[t] == mip.xsum(_yield*U[i][t] for i in range(k)))

# Yield is less than maximum
for t in range(n):
    m.add_constr(W[t] <= max_Y)

### Objective

In [5]:
m.objective = \
    mip.maximize(mip.xsum(price*W[t] for t in range(n)) - # earnings
                 mip.xsum(mip.xsum(storage_c[i]*S[i][t] for i in range(k)) for t in range(1, n + 1)) - # storage costs
                 mip.xsum(mip.xsum(purchase_p[i]*X[i][t] for i in range(k)) for t in range(n))) # purchase costs
m.optimize()

Welcome to the CBC MILP Solver 
Version: Trunk
Build Date: Oct 24 2021 

Starting solution of the Linear programming problem using Dual Simplex



<OptimizationStatus.OPTIMAL: 0>

In [6]:
def print_table(t):
    print('\t', end='')
    for i in range(len(t[0])):
        print(i, end='\t')
    print()
    
    for i in range(len(t)):
        print(i, end='\t')
        for k in range(len(t[i])):
            print(f'{t[i][k]:.2f}', end='\t')
        print()

def print_vec(v):
    for i in range(len(v)):
        print(i, end='\t')
    print()
    for i in range(len(v)):
        print(f'{v[i]:.2f}', end='\t')
    print()
    
def calculate_profits(t):
    earnings = price*W[t].x
    storage_costs = sum(storage_c[i]*S[i][t+1].x for i in range(k))
    purchase_costs = sum(purchase_p[i]*X[i][t].x for i in range(k))
    return earnings - storage_costs - purchase_costs
    

print('--- Total purchased ---')
print_table([[X[i][t].x for t in range(n)] for i in range(k)])

print('\n--- Total taken from storage ---')
print_table([[Y[i][t].x for t in range(n)] for i in range(k)])

print('\n--- Total put into storage ---')
print_table([[Z[i][t].x for t in range(n)] for i in range(k)])

print('\n--- Total stored ---')
print_table([[S[i][t].x for t in range(n+1)] for i in range(k)])

print('\n--- Total energy produced (MWh) ---')
print_vec([W[i].x / 1000 for i in range(n)])

print('\n--- Profits per quarter (k$) ---')
print_vec([calculate_profits(i) / 1000 for i in range(n)])

print('\n--- Objective value ---')
print(m.objective_value)

--- Total purchased ---
	0	1	2	3	4	5	6	7	
0	700.00	1500.00	700.00	0.00	0.00	700.00	1500.00	0.00	
1	1350.00	0.00	450.00	0.00	1350.00	0.00	450.00	0.00	
2	0.00	1500.00	1500.00	0.00	0.00	1500.00	1500.00	0.00	
3	820.00	1560.00	820.00	0.00	820.00	1560.00	820.00	0.00	
4	0.00	565.24	1080.00	0.00	0.00	680.00	855.24	0.00	

--- Total taken from storage ---
	0	1	2	3	4	5	6	7	
0	0.00	-0.00	181.43	318.57	-0.00	0.00	0.00	375.00	
1	0.00	-0.00	0.00	-0.00	0.00	-0.00	0.00	-0.00	
2	0.00	0.00	0.00	106.19	-0.00	0.00	0.00	125.00	
3	0.00	0.00	0.00	-0.00	0.00	0.00	0.00	-0.00	
4	0.00	0.00	0.00	-0.00	-0.00	0.00	0.00	-0.00	

--- Total put into storage ---
	0	1	2	3	4	5	6	7	
0	-0.00	500.00	0.00	0.00	0.00	0.00	375.00	0.00	
1	-0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	
2	-0.00	-0.00	106.19	0.00	0.00	0.00	125.00	0.00	
3	-0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	
4	-0.00	-0.00	0.00	0.00	0.00	0.00	0.00	0.00	

--- Total stored ---
	0	1	2	3	4	5	6	7	8	
0	0.00	0.00	500.00	318.57	0.00	0.00	0.00	375.00	0.00	
1	0.00	0.00	0.00	-0.0